# §38 S0: Nexar source-shortcut probe (CPU only, no training)

Rule: `core/docs/final/KIM_RUN.md` §38, written before any number. Question: can a linear probe on the model's input tell a **normal window inside an accident video** (A, K5f's normals) from a **window of a negative video** (B, the normals K5fno would use)? **S1 = AUC(A vs B) ≥ 0.90 on either read → shortcut, K5fno is not built.**

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | mount, setup + preflight, deps | no | 2-4 min |
| 4 | pytest on the probe — **raises if red** | no | < 1 min |
| 5 | stage `Nexar_window_neg` + the two inputs to the VM (train sources only) | no | 5-15 min |
| 6 | probe on plain CLIP `Nexar_s1_ncc[::8]` and on the A2 V2-S bake | no | 5-10 min |
| 7 | **paste `REPORTS/nexar_source_probe_clip.md` and `..._a2.md` to Claude** | no | — |

Thesis and Thesis-V2 are only read; the reports go to Thesis-Final. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar caches, A2 bakes
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, Nexar_window_neg (§17), every output here
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

REPO = Path(f'{DRIVE_F}/kat-vad')
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert (REPO / 'core/tools/nexar_source_probe.py').is_file(), 'this upload has no nexar_source_probe -- re-upload kat-vad/'
assert '## 38. K5fno' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §38 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

V2_CACHE = Path(DRIVE_V2) / 'cache'
DATA     = Path(DRIVE_F) / 'data' / f'{constants.NEXAR_DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW_NEG}'  # built by k5_neg.ipynb step 5
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET     # plain CLIP, every native frame
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                   # K5f's training input (CLIP + V2-S), s8 rows
STRIDE   = constants.NEXAR_TRAIN_STRIDE                            # corpus rows = s1[::8]
ROOT     = constants.OUTPUT_ROOT / constants.KIM_SOURCE_PROBE_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/source_probe')
LOCAL.mkdir(parents=True, exist_ok=True)
for path in (DATA / constants.META_FILENAME, DATA / constants.WINDOWS_FILENAME, NEXAR_X, A2_NEXAR):
    assert path.exists(), f'missing {path} -- Nexar_window_neg comes from colab/final/k5_neg.ipynb step 5'
print('preflight OK |', DATA, '|', NEXAR_X, '|', A2_NEXAR)


In [ ]:
%%bash
pip install -q scikit-learn
python -c "import sklearn; print('deps OK', sklearn.__version__)"


## 4. Tests on the probe (CPU) — **raises if anything fails: stop and paste the output**


In [ ]:
ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd):
    '''Stream a child's output; raise with its tail on failure.'''
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_nexar_source_probe.py'])


## 5. Stage the corpus and the train sources of both inputs to the VM (bit-identical copies; C2)


In [ ]:
meta = json.loads((DATA / constants.META_FILENAME).read_text())
sources = sorted({str(r['source']) for r in meta.values() if r.get('split') == 'train'})
n_neg = len({str(r['source']) for r in meta.values() if r.get('split') == 'train' and not r['source_is_abnormal']})
print(f'{len(sources)} train sources ({n_neg} negative videos)')
assert n_neg > 0, 'no negative-video window: this is not Nexar_window_neg'

local_data = LOCAL / 'data'
local_data.mkdir(exist_ok=True)
for f in (constants.META_FILENAME, constants.WINDOWS_FILENAME):
    shutil.copy2(DATA / f, local_data / f)


def stage(src, dst):
    t0 = time.time()
    dst.mkdir(parents=True, exist_ok=True)
    missing = [v for v in sources if not (src / f'{v}.npy').exists()]
    assert not missing, f'{len(missing)} train sources missing in {src}: {missing[:3]}'
    for v in sources:
        if not (dst / f'{v}.npy').exists():
            shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
    print(f'staged {len(sources)} files {src.name} -> {dst} in {time.time() - t0:.0f} s')
    return dst


CLIP_LOCAL = stage(NEXAR_X, LOCAL / 'clip_s1')
A2_LOCAL = stage(A2_NEXAR, LOCAL / 'a2_s8')


## 6. S0 probe on both reads (§38 S0-2)


In [ ]:
out = LOCAL / 'reports'
for name, clip_dir, stride in (('clip', CLIP_LOCAL, STRIDE), ('a2', A2_LOCAL, 1)):
    run([sys.executable, '-m', 'core.tools.nexar_source_probe', '--data-dir', local_data, '--clip-dir', clip_dir,
         '--row-stride', stride, '--name', name, '--out-dir', out])
REPORTS.mkdir(parents=True, exist_ok=True)
verdicts = {}
for name in ('clip', 'a2'):
    stem = f'{constants.NEXAR_SOURCE_PROBE_STEM}_{name}'
    for ext in ('.json', '.md'):
        shutil.copy2(out / f'{stem}{ext}', REPORTS / f'{stem}{ext}')
    verdicts[name] = json.loads((out / f'{stem}.json').read_text())['shortcut']
    print((out / f'{stem}.md').read_text())
print('S0 gate (§38 S0-4):', 'SHORTCUT -- K5fno is not built' if any(verdicts.values()) else 'passed on both reads', verdicts)
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/nexar_source_probe_clip.md and nexar_source_probe_a2.md to Claude')
